# Reality Engine — Portable Colab Fallback (when away from RX 6700 XT machine)
One-stop interface fallback: runs same `postgres_schema.sql 748 lines` funnel + `dashboard 8-tab` in Colab. Uses SQLite WAL fallback when PG not provisioned (Colab has no local PG). GPU fallback: Colab T4 → RapidOCR CPU, Whisper base.
Kilo CLI not needed — notebook is one-stop.

In [ ]:
!git clone https://github.com/your-org/Info_Diffusion_Model.git
%cd Info_Diffusion_Model
!pip install -q -r requirements.txt rapidocr_onnxruntime yt-dlp openai-whisper lancedb
!pip install -q streamlit
!python reality_engine/db/seed_fundamental.py  # countries 3, industries 12 →10≥4.0
import sqlite3; print(list(sqlite3.connect('reality_engine/data/equity_intelligence.db').execute('SELECT count(*) FROM master_companies'))[0][0])


In [ ]:
# Historical fetch — mirrors local one-stop dashboard -> same pipeline
# Price/volume max period (1250 sessions ≈5y) — NSE+BSE Bhavcopy, workers=8, rate-limited
!python reality_engine/cli.py backfill --days 1250 --workers 8
# Financials 5y for all 2730 ISINs
!python reality_engine/cli.py fetch-fundamentals --years 5 --workers 8
# Verify
!python -c "import sqlite3; db='reality_engine/data/equity_intelligence.db'; con=sqlite3.connect(db); cur=con.cursor(); print('daily',cur.execute('SELECT count(*) FROM daily_price_delivery').fetchone()[0]); print('quarterly',cur.execute('SELECT count(*) FROM quarterly_financials').fetchone()[0])"


In [ ]:
# Launch dashboard as one-stop interface (Colab public URL)
!streamlit run reality_engine/ui/dashboard.py --server.port 8501 --server.headless true &
from google.colab import output; output.serve_kernel_port_as_window(8501)
# Use tabs 1-8: Graph Explorer PN/MN/S, Serendipity Inbox drag likes/, Moat Lab, Policy Radar S(t), Visual Alpha 48jpg, Distillation Log